# 06 — Обучение групповых агрегаторов

Вся логика — в `research/scripts/train_aggregators.py` и конфиге
`research/configs/aggregators_50m.yaml`. Здесь только запуск и графики.

Смок локально: `--config research/configs/smoke_aggregators.yaml`.

In [ ]:
# Colab: раскомментировать. Локально ячейка не нужна.
# from google.colab import userdata
# token = userdata.get('git')
# !git clone -q https://$token@github.com/Vladislavbro/music-recommendations.git
# %cd music-recommendations
# !pip install -q uv && uv pip install --system -e ".[research]"


In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download

HF_REPO = 'Vladislavbro-500/music-recommendations'
ARTIFACTS = Path.cwd() / 'artifacts'

NEEDED = [
    'gsasrec/item_id_to_idx.pkl',
    'user_scores_cache/scores.parquet',
    'audio/embeddings.npy',
    'audio/user_profiles.npy',
    'audio/uid_to_row.pkl',
]
for rel in NEEDED:
    if (ARTIFACTS / rel).exists():
        continue
    hf_hub_download(repo_id=HF_REPO, repo_type='dataset', filename=rel, local_dir=str(ARTIFACTS))
print('artifacts ready')


## Запуск

Чекпоинты, `metrics.csv`, `config.resolved.json` и `run.json` пишутся в `artifacts/aggregators/<метод>/`.

In [ ]:
CONFIG = 'research/configs/aggregators_50m.yaml'

!uv run python research/scripts/train_aggregators.py --config {CONFIG}


## Кривые обучения

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import yaml

models = yaml.safe_load(open(CONFIG))['models']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key, spec in models.items():
    path = ARTIFACTS / 'aggregators' / key / 'metrics.csv'
    if not path.exists():
        continue
    m = pd.read_csv(path)
    axes[0].plot(m['epoch'], m['train_loss'], marker='o', markersize=3, label=spec['name'])
    axes[1].plot(m['epoch'], m['val_NDCG@10'], marker='o', markersize=3, label=spec['name'])

for ax, title in zip(axes, ['Train BPR loss', 'Val NDCG@10']):
    ax.set_title(title); ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Залить чекпоинты обратно на HF

In [ ]:
# Colab: раскомментировать.
# import os
# from google.colab import userdata
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# !hf upload Vladislavbro-500/music-recommendations \
#     artifacts/aggregators aggregators \
#     --type dataset --commit-message 'aggregator checkpoints'
